In [1]:
import fsspec

In [2]:
fsspec.exists

AttributeError: module 'fsspec' has no attribute 'exists'

In [3]:
path = "gs://marin-us-central1/checkpoints/math_sft_sweep/math500_sft/-06)_warmup_VersionedValue(value=0.1)-2ba30e/checkpoints/eval_metrics.jsonl"

with fsspec.open(path, "rt") as f:
    for line in f:
        print(line)





/Users/rohith/research/marin/.venv/lib/python3.11/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


{"eval/Qwen--Qwen3-0-6B/bpb": 0.4613361656665802, "eval/Qwen--Qwen3-0-6B/loss": 0.7770224213600159, "eval/bpb": 0.4613361656665802, "eval/loading_time": 0.1182834109858959, "eval/loss": 0.7770224213600159, "eval/total_time": 4.272844254999654, "step": 999}



In [9]:
import gcsfs                                                                              
                                                                                            
fs = gcsfs.GCSFileSystem()                                                                
test_path = "gs://marin-us-central1/tmp/test_append.txt"                                
                                                                                        
with fs.open(test_path, "w") as f:                                                        
    f.write("line1\n")                                                                  
                                                            
with fs.open(test_path, "r") as f:
    content = f.read()
    content += "line2\n"

with fs.open(test_path, "w") as f:
    f.write(content)

with fs.open(test_path, "r") as f:
    print(f.read())

fs.rm(test_path)

line1
line2



['marin-us-central1/tmp/test_append.txt']

In [18]:
from experiments.evals.test_save_logprobs import (
    TOY_DOCUMENTS,
    WriteToyDataConfig,
    write_toy_data,
    toy_data_step,
    tokenize_step,
    eval_data,
    model_info,
    model_instance,
    model_identifier,
    hf_model_config,
    save_logprobs_step,
)
from marin.execution.executor import Executor, ExecutorStep, this_output_path

import os
import json

def get_step_output_path(step: ExecutorStep, prefix: str) -> str:
    """
    Get the output path for an ExecutorStep.

    Args:
        step: The ExecutorStep to get the output path for
        prefix: The prefix to use. If None, uses MARIN_PREFIX env var.

    Returns:
        The output path as a string.
    """

    executor_info_base_path = os.path.join(prefix, "experiments")
    executor = Executor(
        prefix=prefix,
        executor_info_base_path=executor_info_base_path,
    )
    executor.compute_version(step, is_pseudo_dep=False)
    return executor.output_paths[step]


In [33]:
gcs_path = get_step_output_path(save_logprobs_step, "gs://marin-us-east5")
file_path = os.path.join(gcs_path, "toy.jsonl.gz")

from transformers import AutoTokenizer                                                    
                                                                                            
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")  

with fsspec.open(file_path, "rt", compression="gzip") as f:
    for line in f:
        data = json.loads(line)
        print(data.keys())

        for i in range(5):
            print(data["token_logprobs"][i])
            print(data["token_ids"][i])
            print(data["token_ids"][i+1])
            print(data["top_k_logprobs"][i])
            print(data["top_k_token_ids"][i])

        print(tokenizer.decode(data["token_ids"], skip_special_tokens=True))

        print(type(data["token_logprobs"]))





2026-02-28 20:21:39,340	WARNING executor.py:829 -- Output path gs://marin-us-east5/models/meta-llama--Llama-3-2-1B--main-e2de2b doesn't match given override models/meta-llama--Llama-3-2-1B--main, using the latter.


dict_keys(['token_ids', 'token_logprobs', 'top_k_token_ids', 'top_k_logprobs'])
-3.616013526916504
128000
791
[-1.209769606590271, -2.223127841949463, -2.698181629180908, -3.616014003753662, -4.317171573638916, -4.398285388946533, -4.476412296295166, -4.793476581573486, -4.958180904388428, -4.9662909507751465]
[14924, 755, 2, 791, 16309, 475, 32, 7778, 2028, 40]
-9.268936157226562
791
4062
[-3.711907386779785, -4.286005020141602, -4.647116661071777, -5.130129814147949, -5.217413902282715, -5.272880554199219, -5.31352424621582, -5.347823143005371, -5.438455581665039, -5.4627790451049805]
[220, 2768, 1176, 502, 7252, 1888, 1455, 1561, 5165, 549]
-4.853354454040527
4062
14198
[-1.7294458150863647, -2.6440672874450684, -3.449796199798584, -3.521165370941162, -3.7652831077575684, -3.8197760581970215, -3.992419719696045, -4.178704738616943, -4.225112438201904, -4.259155750274658]
[323, 4320, 11, 1648, 8641, 6147, 1212, 2077, 2373, 92346]
-0.04804039001464844
14198
39935
[-0.04803988337516785

In [24]:
len(TOY_DOCUMENTS[0])

2250

In [34]:
from datasets import load_dataset, concatenate_datasets

configs = ["algebra", "counting_and_probability", "geometry",                             
             "intermediate_algebra", "number_theory", "prealgebra", "precalculus"]
dataset = concatenate_datasets([
    load_dataset("HuggingFaceH4/MATH", config, split="train")
    for config in configs
])
rows = [dict(row) for row in dataset]

Generating test split: 100%|██████████| 546/546 [00:00<00:00, 258387.68 examples/s]


In [39]:
len(rows[0]['solution'])

352

In [40]:
from datasets import load_dataset, concatenate_datasets

configs = ["algebra", "counting_and_probability", "geometry",                             
             "intermediate_algebra", "number_theory", "prealgebra", "precalculus"]
dataset = concatenate_datasets([
    load_dataset("HuggingFaceH4/MATH", config, split="test")
    for config in configs
])
rows = [dict(row) for row in dataset]

In [41]:
len(rows)

5000

In [42]:
import numpy as np

x = np.random.normal(size=(10,20,30))

x[..., 0].shape

(10, 20)

In [44]:
path = "gs://marin-us-central2/analysis/logprobs/math_sft_sweep/math500_base_logprobs/Qwen--Qwen3-0-6B-f11a2b/test/outputs.jsonl.gz"

with fsspec.open(path, "rt", compression="gzip") as f:
    i = 0
    for line in f:
        i += 1

print(i)

500
